# 5x1000 — Beneficiari e importi per ente (2023-2025)

Dataset: `ade_cinque_per_mille` (Agenzia delle Entrate)

Analisi pubblica: [README](../README.md)

In [1]:
import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FIGS = Path('../figures')
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute("SET s3_region='us-east-1';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")
con.execute("SET s3_session_token='';")

GCS_PATH = 'gs://dataciviclab-clean/ade_cinque_per_mille/*/ade_cinque_per_mille_*_clean.parquet'
print('DuckDB ready, GCS anonymous access')

DuckDB ready, GCS anonymous access


In [2]:
# 0. Overview: trend 2023-2025
df_trend = con.execute(f"""
    SELECT anno,
           COUNT(*) AS n_enti,
           ROUND(SUM(importo_totale_erogabile) / 1e6, 0) AS importo_milioni,
           ROUND(SUM(numero_scelte) / 1e6, 1) AS scelte_milioni
    FROM '{GCS_PATH}'
    GROUP BY anno
    ORDER BY anno
""").fetchdf()

print('Trend 2023-2025:')
print(df_trend.to_string(index=False))

Trend 2023-2025:
 anno  n_enti  importo_milioni  scelte_milioni
 2023   80597            521.0            14.4
 2024   90611            522.0            15.1
 2025   95977            601.0            15.5


In [3]:
# Figura 0: Trend
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

colors_t = ['#2c3e50', '#2c3e50', '#27ae60']

bars1 = ax1.bar(df_trend['anno'].astype(str), df_trend['importo_milioni'], color=colors_t, edgecolor='white', width=0.5)
for bar, val in zip(bars1, df_trend['importo_milioni']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('Importo totale erogabile', fontweight='bold')
ax1.set_ylabel('Milioni di €')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

bars2 = ax2.bar(df_trend['anno'].astype(str), df_trend['n_enti'], color=colors_t, edgecolor='white', width=0.5)
for bar, val in zip(bars2, df_trend['n_enti']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 800,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Enti beneficiari', fontweight='bold')
ax2.set_ylabel('Numero enti')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('5x1000 — Trend 2023-2025', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'cinque-per-mille_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: cinque-per-mille_trend.png')

Figura salvata: cinque-per-mille_trend.png


In [4]:
# 1. Categorie per anno (2023 vs 2025)
df_cat = con.execute(f"""
    SELECT anno,
           CASE
               WHEN flag_ets_onlus AND NOT flag_asd AND NOT flag_comune
                    AND NOT flag_ricerca_scientifica AND NOT flag_ricerca_sanitaria THEN 'ETS / ONLUS'
               WHEN flag_ets_onlus AND flag_ricerca_scientifica AND flag_ricerca_sanitaria THEN 'ETS + Ricerca scient. e san.'
               WHEN flag_ets_onlus AND flag_ricerca_scientifica AND NOT flag_ricerca_sanitaria THEN 'ETS + Ricerca scientifica'
               WHEN flag_ets_onlus AND NOT flag_ricerca_scientifica AND flag_ricerca_sanitaria THEN 'ETS + Ricerca sanitaria'
               WHEN flag_ets_onlus AND flag_asd THEN 'ETS + ASD'
               WHEN flag_ets_onlus AND flag_comune THEN 'ETS + Comune'
               WHEN flag_asd AND NOT flag_ets_onlus THEN 'Sportive dilettantistiche'
               WHEN flag_ricerca_scientifica AND flag_ricerca_sanitaria AND NOT flag_ets_onlus THEN 'Ricerca scientifica e sanitaria'
               WHEN flag_ricerca_scientifica AND NOT flag_ricerca_sanitaria AND NOT flag_ets_onlus THEN 'Ricerca scientifica'
               WHEN NOT flag_ricerca_scientifica AND flag_ricerca_sanitaria AND NOT flag_ets_onlus THEN 'Ricerca sanitaria'
               WHEN flag_comune AND NOT flag_ets_onlus THEN 'Comuni'
               WHEN flag_beni_culturali THEN 'Beni culturali'
               WHEN flag_area_protetta THEN 'Aree protette'
               ELSE 'Altro (senza categoria assegnata)'
           END AS categoria,
           COUNT(*) AS n_enti,
           ROUND(SUM(importo_totale_erogabile) / 1e6, 1) AS importo_milioni
    FROM '{GCS_PATH}'
    WHERE anno IN (2023, 2025)
    GROUP BY anno, categoria
    ORDER BY anno, importo_milioni DESC
""").fetchdf()

print('Categorie 2023 vs 2025:')
for anno in [2023, 2025]:
    print(f'\n--- {anno} ---')
    sub = df_cat[df_cat['anno'] == anno]
    print(sub[['categoria', 'n_enti', 'importo_milioni']].to_string(index=False))

Categorie 2023 vs 2025:

--- 2023 ---
                      categoria  n_enti  importo_milioni
                    ETS / ONLUS   58694            310.5
   ETS + Ricerca Scient. e San.       9             93.6
              Ricerca sanitaria      86             36.2
            Ricerca scientifica     426             26.7
Ricerca scientifica e sanitaria      11             19.9
      Sportive dilettantistiche   13306             17.9
                         Comuni    7909             15.3
                  Aree protette      21              0.5
                 Beni culturali     135              0.5

--- 2025 ---
                      categoria  n_enti  importo_milioni
                    ETS / ONLUS   70366            326.3
   ETS + Ricerca Scient. e San.       8            110.3
              Ricerca sanitaria      82             39.6
            Ricerca scientifica     459             34.0
Ricerca scientifica e sanitaria      17             27.1
                          Altro    2

In [5]:
# Figura 1: Categorie 2025
cat_2025 = df_cat[df_cat['anno'] == 2025].sort_values('importo_milioni', ascending=True)
palette = ['#2c3e50', '#27ae60', '#e74c3c', '#95a5a6', '#f39c12', '#3498db', '#8e44ad', '#1abc9c', '#d35400', '#7f8c8d']

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(cat_2025['categoria'], cat_2025['importo_milioni'], color=palette[:len(cat_2025)], edgecolor='white')
ax.set_xlabel('Milioni di euro')
ax.set_title('5x1000 per categoria — 2025', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val, enti in zip(bars, cat_2025['importo_milioni'], cat_2025['n_enti']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'€{val:.1f}M — {enti:,} enti', va='center', fontsize=9)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f} M'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGS / 'cinque-per-mille_categorie.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: cinque-per-mille_categorie.png')

Figura salvata: cinque-per-mille_categorie.png


In [6]:
# 2. Regioni 2025
reg_2025 = con.execute(f"""
    SELECT regione,
           COUNT(*) AS n_enti,
           ROUND(SUM(importo_totale_erogabile) / 1e6, 1) AS importo_milioni
    FROM '{GCS_PATH}'
    WHERE anno = 2025 AND regione IS NOT NULL AND regione != ''
    GROUP BY regione
    ORDER BY importo_milioni DESC
""").fetchdf()

print('Regioni 2025:')
print(reg_2025.to_string(index=False))

Regioni 2025:
                               regione  n_enti  importo_milioni
                             LOMBARDIA   15889            215.3
                                 LAZIO    9015            112.8
                        EMILIA ROMAGNA    8242             39.8
                              PIEMONTE    8426             36.9
                                VENETO    8103             35.8
                               TOSCANA    6090             24.8
                               LIGURIA    2404             23.5
                              CAMPANIA    6015             18.6
                                MARCHE    2637             18.0
                                PUGLIA    5453             16.5
                               SICILIA    5986             15.1
                 FRIULI VENEZIA GIULIA    2618             10.3
         TRENTINO-ALTO ADIGE/SUEDTIROL    3555             10.0
                              CALABRIA    3046              5.8
                          

In [7]:
# Figura 2: Regioni 2025
fig, ax = plt.subplots(figsize=(12, 6.5))
med = reg_2025['importo_milioni'].median()
colors_r = ['#2c3e50' if v >= med else '#95a5a6' for v in reg_2025['importo_milioni']]

bars = ax.barh(reg_2025['regione'], reg_2025['importo_milioni'], color=colors_r, edgecolor='white')
ax.set_xlabel('Milioni di euro')
ax.set_title('5x1000 per regione — 2025', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, reg_2025['importo_milioni']):
    if val and not np.isnan(val):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'€{val:.1f}M', va='center', fontsize=8)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f} M'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGS / 'cinque-per-mille_regioni.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: cinque-per-mille_regioni.png')

Figura salvata: cinque-per-mille_regioni.png


In [8]:
# 3. Top 10 2025
top10 = con.execute(f"""
    SELECT denominazione, regione, numero_scelte,
           ROUND(importo_totale_erogabile / 1e6, 1) AS importo_milioni
    FROM '{GCS_PATH}'
    WHERE anno = 2025
    ORDER BY importo_totale_erogabile DESC NULLS LAST
    LIMIT 10
""").fetchdf()

print('Top 10 2025:')
print(top10[['denominazione', 'regione', 'importo_milioni']].to_string(index=False))

Top 10 2025:
                                                                          denominazione   regione  importo_milioni
                                          FONDAZIONE AIRC PER LA RICERCA SUL CANCRO ETS LOMBARDIA             82.7
                                    FONDAZIONE PIEMONTESE PER LA RICERCA SUL CANCRO ETS  PIEMONTE             14.3
 EMERGENCY - LIFE SUPPORT FOR CIVILIAN WAR VICTIMS - ONG ETS IN BREVE EMERGENCY ONG ETS LOMBARDIA             13.4
                            FONDAZIONE LEGA DEL FILO D'ORO - E.T.S. - ENTE FILANTROPICO    MARCHE             11.3
AIL - ASSOCIAZIONE ITALIANA CONTRO LE LEUCEMIE-LINFOMI E MIELOMA ENTE DEL TERZO SETTORE     LAZIO             10.2
                                                   ISTITUTO EUROPEO DI ONCOLOGIA S.R.L. LOMBARDIA              9.5
                                                           MEDICI SENZA FRONTIERE ONLUS     LAZIO              9.4
                       FONDAZIONE ITALIANA SCLEROSI MULTIPLA ENTE D

In [9]:
# Figura 3: Top 10
fig, ax = plt.subplots(figsize=(12, 5.5))
yticks = range(len(top10))
labels_top = [d[:50] + '...' if len(d) > 50 else d for d in top10['denominazione']]
bars = ax.barh(yticks, top10['importo_milioni'], color='#2c3e50', edgecolor='white')
ax.set_yticks(yticks)
ax.set_yticklabels(labels_top, fontsize=9)
ax.set_xlabel('Milioni di euro')
ax.set_title('Top 10 beneficiari 5x1000 — 2025', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, top10['importo_milioni']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'€{val:.1f}M', va='center', fontsize=10, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGS / 'cinque-per-mille_top20.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: cinque-per-mille_top20.png')

Figura salvata: cinque-per-mille_top20.png


In [10]:
# Riepilogo
totali = con.execute(f"""
    SELECT anno, COUNT(*) AS n_enti,
           SUM(numero_scelte) AS tot_scelte,
           ROUND(SUM(importo_totale_erogabile) / 1e6, 0) AS importo_milioni
    FROM '{GCS_PATH}'
    GROUP BY anno
    ORDER BY anno
""").fetchdf()

print('RIEPILOGO — 5x1000 2023-2025')
for _, row in totali.iterrows():
    print(f"{int(row['anno'])}: {int(row['n_enti']):,} enti, €{int(row['importo_milioni'])}M, {row['tot_scelte']/1e6:.1f}M scelte")

print(f"\nTotale 2023-2025: €{totali['importo_milioni'].sum():.0f}M")
print(f"Variazione 2023-2025: {(totali[totali.anno==2025]['importo_milioni'].values[0] / totali[totali.anno==2023]['importo_milioni'].values[0] - 1)*100:.1f}%")

RIEPILOGO — 5x1000 2023-2025
2023: 80,597 enti, €521M, 14.4M scelte
2024: 90,611 enti, €522M, 15.1M scelte
2025: 95,977 enti, €601M, 15.5M scelte

Totale 2023-2025: €1644M
Variazione 2023-2025: 15.4%


In [11]:
print('Tutte le figure generate:')
for f in sorted(FIGS.glob('cinque-per-mille_*.png')):
    print(f'  ✓ {f.name}')

Tutte le figure generate:
  ✓ cinque-per-mille_categorie.png
  ✓ cinque-per-mille_fasce.png
  ✓ cinque-per-mille_regioni.png
  ✓ cinque-per-mille_top20.png
  ✓ cinque-per-mille_trend.png
